### Model View Conroller


In [69]:
from ipywidgets import Output
from ipycanvas import MultiCanvas
from IPython.display import display


class Observable:
    def __init__(self):
        self._callbacks = {}

    def observe(self, fun):
        'Funktion fun als Callback registrieren'
        self._callbacks[fun.__name__] = fun

    def unobserve(self, fun):
        'registiertes Callback entfernen'
        if fun.__name__ in self._callbacks:
            self._callbacks.pop(fun.__name__)

    def _notify(self, event, **kwargs):
        '''ruft alle registrierten Callbacks f mit
           f(event, **kwargs) auf
        '''
        for f in self._callbacks.values():
            f(event, **kwargs)


class Game(Observable):
    def new_game(self):
        print('starting a new game')
        self._notify('new_game')

    def move(self, arg):
        print(f'making a move {arg}')
        self._notify('move', arg=arg)


class View:
    layout = {'border': '1px solid black'}

    def __init__(self, game=None, *, width=200, height=200):
        self.game = game
        if isinstance(game, Observable):
            self.game.observe(self.update)

        self.width = width
        self.height = height
        self.mcanvas = MultiCanvas(3, width=self.width, height=self.height, layout=self.layout)
        self.bg, self.fg, self.info = self.mcanvas
        self.info.text_align = 'center'
        self.info.text_baseline = 'middle'
        self.info.font = f'{self.height//3}px sans-serif'

    def update(self, event, **kwargs):
        getattr(self, event)(**kwargs)

    def new_game(self):
        self.mcanvas.clear()
        self.info.fill_text('New Game!', self.width/2, self.height/2, max_width=self.width)

    def move(self, arg):
        self.mcanvas.clear()
        self.info.fill_text(f'Move {arg}!', self.width/2, self.height/2, max_width=self.width)

    def _ipython_display_(self):
        display(self.mcanvas)


class Controller:
    layout = {'border': '1px solid black'}
    out = Output(layout=layout)

    def __init__(self, canvas, game):
        self.game = game
        self.canvas = canvas
        self.init_controller()

    def init_controller(self):
        self.canvas.on_mouse_down(self.on_mouse_down)
        self.canvas.on_mouse_up(self.on_mouse_up)
        self.canvas.on_mouse_out(self.on_mouse_out)
        self.canvas.on_mouse_move(self.on_mouse_move)
        self.canvas.on_key_down(self.on_key_down)

    @out.capture(clear_output=True)
    def on_mouse_down(self, x, y):
        print(f'mouse down at ({int(x)}, {int(x)})')

    @out.capture(clear_output=True)
    def on_mouse_up(self, x, y):
        print(f'mouse up at ({int(x)}, {int(x)})')

    @out.capture(clear_output=True)
    def on_mouse_out(self, x, y):
        print(f'mouse out at ({int(x)}, {int(x)})')

    @out.capture(clear_output=True)
    def on_mouse_move(self, x, y):
        print(f'mouse move at ({int(x)}, {int(x)})')

    @out.capture(clear_output=True)
    def on_key_down(self, key, *flags):
        print(f'key "{key}" down')
        if key == 'n':
            self.game.new_game()
        elif key.startswith('Arrow'):
            game.move(key[5:])

    def _ipython_display_(self):
        display(self.canvas, self.out)

In [70]:
game = Game()
view = View(game)
Controller(view.mcanvas, game)

MultiCanvas(height=200, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [63]:
game.new_game()

starting a new game


In [64]:
game.move()

making a move


In [65]:
game._callbacks

{'update': <bound method View.update of <__main__.View object at 0x7f01ec0dc050>>}